# Kết quả và đối chiếu các mô hình

Notebook này **chỉ đọc kết quả** mà các script đã lưu trong `results/` rồi hiển thị cạnh nhau, không huấn luyện lại. Muốn số mới thì chạy lại script, rồi chạy lại notebook.

Chạy các lệnh sau từ thư mục gốc repo trước khi mở notebook:

```bash
python -m experiments.run_baseline
python -m experiments.run_svd
python -m experiments.run_als
python -m experiments.run_sgd
python -m experiments.compare_all
```

Mục nào thiếu file thì notebook in ra cần chạy lệnh nào, không báo lỗi.

## 0. Chuẩn bị

In [ ]:
import os
import sys
from pathlib import Path

# Notebook nằm trong notebooks/, nên chuyển về gốc repo để `import src` và đường dẫn tương đối hoạt động.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.config import load_config

cfg = load_config()
TABLES = cfg['paths']['tables_dir']
FIGURES = cfg['paths']['figures_dir']
MODELS = cfg['paths']['models_dir']
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
pd.set_option('display.width', 200)


def read_table(name, command):
    """Đọc results/tables/<name>; thiếu thì in lệnh cần chạy và trả về None."""
    path = TABLES / name
    if not path.exists():
        print(f'Chưa có {path.name}. Chạy: python -m {command}')
        return None
    return pd.read_csv(path)


def show_figure(name, command):
    """Hiển thị results/figures/<name>; thiếu thì in lệnh cần chạy."""
    path = FIGURES / name
    if path.exists():
        display(Image(filename=str(path), width=900))
    else:
        print(f'Chưa có {path.name}. Chạy: python -m {command}')


print('Gốc repo:', ROOT)
print('Mô hình đã lưu:', sorted(p.name for p in MODELS.glob('*.pkl')) if MODELS.exists() else 'chưa có')

## 1. Bảng chính: RMSE trên tập test

Lấy từ `compare_all.py`, nơi duy nhất dùng tập test. Cột `vs_baseline` là phần trăm RMSE giảm so với Bias baseline: mô hình phân rã phải dương thì mới có ý nghĩa.

In [ ]:
test = read_table('compare_test.csv', 'experiments.compare_all')
if test is not None:
    base = test.loc[test['model'] == 'Bias baseline', 'test_rmse']
    if len(base):
        test['vs_baseline_%'] = 100 * (base.iloc[0] - test['test_rmse']) / base.iloc[0]
    display(test.sort_values('test_rmse').reset_index(drop=True))

In [ ]:
# Biểu đồ cột vẽ ngay trong notebook (chưa phải hình chính thức của báo cáo, hình đó do plot_bars vẽ).
if test is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, column, label in ((axes[0], 'test_rmse', 'Test RMSE'), (axes[1], 'fit_time_s', 'Training time (s)')):
        bars = ax.bar(test['model'], test[column], color='#2a78d6')
        ax.bar_label(bars, fmt='%.4f' if column == 'test_rmse' else '%.2f')
        ax.set_ylabel(label)
        ax.tick_params(axis='x', rotation=20)
    axes[0].set_ylim(0.8, test['test_rmse'].max() * 1.03)
    axes[1].set_yscale('log')
    plt.tight_layout()
    plt.show()

## 2. Val và test cạnh nhau

Val dùng để chọn tham số, test chỉ để báo cáo. Hai cột gần nhau nghĩa là việc chọn tham số không bị "học thuộc" tập val.

In [ ]:
rows = []
baseline = read_table('baseline.csv', 'experiments.run_baseline')
if baseline is not None:
    for _, r in baseline.iterrows():
        name = {'GlobalMean': 'Global mean', 'BiasBaseline': 'Bias baseline'}.get(r['model'], r['model'])
        rows.append({'model': name, 'val_rmse': r['val_rmse']})
svd = read_table('svd_metrics.csv', 'experiments.run_svd')
if svd is not None:
    rows.append({'model': 'SVD', 'val_rmse': svd['val_rmse'].iloc[0]})
als_log = read_table('als_convergence.csv', 'experiments.run_als')
if als_log is not None:
    rows.append({'model': 'ALS', 'val_rmse': als_log['val_rmse'].iloc[-1]})
sgd = read_table('sgd_comparison.csv', 'experiments.run_sgd')
if sgd is not None:
    best = sgd.sort_values('val_rmse').iloc[0]
    rows.append({'model': f"SGD ({best['optimizer']})", 'val_rmse': best['val_rmse']})

side = pd.DataFrame(rows)
if test is not None and len(side):
    side = side.merge(test[['model', 'test_rmse']], on='model', how='left')
    side['test_minus_val'] = side['test_rmse'] - side['val_rmse']
display(side)

## 3. Hội tụ: ALS và SGD

Hình từ `compare_all.py`. Baseline và SVD không lặp nên là đường ngang. Hình theo thời gian quan trọng hơn hình theo vòng lặp, vì một vòng ALS và một epoch SGD tốn thời gian khác nhau.

In [ ]:
show_figure('compare_convergence_epoch.png', 'experiments.compare_all')
show_figure('compare_convergence_elapsed_s.png', 'experiments.compare_all')

Bảng dưới tìm **vòng có val RMSE thấp nhất** của từng mô hình lặp. Nếu `best_epoch` nhỏ hơn nhiều so với `last_epoch` và `last_val` cao hơn `best_val` rõ rệt, mô hình đang overfit: nên giảm số epoch hoặc tăng `reg`.

In [ ]:
histories = {'ALS': ('als_convergence.csv', 'experiments.run_als')}
for opt in ('sgd', 'momentum', 'adam'):
    histories[f'SGD ({opt})'] = (f'sgd_{opt}_history.csv', 'experiments.run_sgd')

summary = []
for name, (file, command) in histories.items():
    h = read_table(file, command)
    if h is None or h['val_rmse'].isna().all():
        continue
    i = h['val_rmse'].idxmin()
    summary.append({
        'model': name,
        'best_epoch': int(h.loc[i, 'epoch']), 'best_val': h.loc[i, 'val_rmse'],
        'time_to_best_s': h.loc[i, 'elapsed_s'],
        'last_epoch': int(h['epoch'].iloc[-1]), 'last_val': h['val_rmse'].iloc[-1],
        'total_time_s': h['elapsed_s'].iloc[-1],
    })
summary = pd.DataFrame(summary)
if len(summary):
    summary['overfit_gap'] = summary['last_val'] - summary['best_val']
display(summary)

## 4. SGD: so sánh 3 optimizer (cùng k, λ)

In [ ]:
if sgd is not None:
    display(sgd)
show_figure('sgd_convergence_epoch.png', 'experiments.run_sgd')
show_figure('sgd_convergence_elapsed_s.png', 'experiments.run_sgd')

## 5. SVD: phổ giá trị kỳ dị

σ giảm nhanh nghĩa là ma trận phần dư xấp xỉ được bằng hạng thấp, là cơ sở của phân rã ma trận (Ch.4).

In [ ]:
if svd is not None:
    display(svd)
show_figure('svd_spectrum.png', 'experiments.run_svd')

## 6. ALS: số điều kiện theo λ

κ₂ của Q_uᵀQ_u + λI giảm khi λ tăng: λ vừa chống overfit vừa làm hệ dễ giải hơn (Ch.2, Ch.3). `kappa_max` lớn ở λ nhỏ đến từ user có ít đánh giá.

In [ ]:
cond = read_table('als_conditioning.csv', 'experiments.run_als')
if cond is not None:
    display(cond.style.format({'reg': '{:g}', 'kappa_median': '{:.2f}', 'kappa_max': '{:.1f}'}))
show_figure('als_condition_vs_lambda.png', 'experiments.run_als')

## 7. Đối chiếu dự đoán trên vài đánh giá của tập test

Lấy ngẫu nhiên vài đánh giá thật trong test và xem từng mô hình đoán bao nhiêu. Đổi `N_SAMPLES` hoặc `SEED` để xem mẫu khác.

In [ ]:
from src.data.load import load_movies
from src.data.preprocess import load_processed
from src.models.base import Recommender

N_SAMPLES, SEED = 10, 0

split = load_processed(cfg)
movies = load_movies(cfg['paths']['raw_dir']).set_index('movie_id')
model_files = {'Bias baseline': 'baseline.pkl', 'SVD': 'svd.pkl', 'ALS': 'als.pkl', 'SGD': 'sgd_best.pkl'}
models = {name: Recommender.load(MODELS / f) for name, f in model_files.items() if (MODELS / f).exists()}
print('Đọc được:', list(models))

sample = split.test.sample(N_SAMPLES, random_state=SEED)
users, items = sample['user_idx'].to_numpy(), sample['item_idx'].to_numpy()
compare = pd.DataFrame({
    'user_id': split.maps.user_ids[users],
    'title': movies.loc[split.maps.movie_ids[items], 'title'].to_numpy(),
    'actual': sample['rating'].to_numpy(),
})
for name, model in models.items():
    compare[name] = model.predict(users, items)
display(compare)

## 8. Top 10 gợi ý của từng mô hình cho cùng một user

Bias baseline xếp hạng chỉ theo b_i, nên mọi user nhận gần như cùng một danh sách. Mô hình phân rã học thêm p_uᵀq_i nên danh sách khác nhau theo user. Đổi `USER_ID` để thử.

In [ ]:
from src.recommend import recommend_for_user

USER_ID = 42

movies_df = movies.reset_index()
tops = {}
for name, model in models.items():
    tops[name] = recommend_for_user(model, USER_ID, split.maps, movies_df, n=10)
display(pd.DataFrame({name: t['title'].to_numpy() for name, t in tops.items()}, index=range(1, 11)))

# Số phim trùng giữa từng cặp danh sách: trùng nhiều với baseline nghĩa là mô hình ít cá nhân hóa.
names = list(tops)
overlap = pd.DataFrame(
    [[len(set(tops[a]['title']) & set(tops[b]['title'])) for b in names] for a in names],
    index=names, columns=names,
)
display(overlap)

Phim user này đã đánh giá cao trong train, để đối chiếu với gợi ý ở trên:

In [ ]:
u = split.maps.user_index[USER_ID]
seen = split.train[split.train['user_idx'] == u].sort_values('rating', ascending=False).head(10)
display(pd.DataFrame({
    'title': movies.loc[split.maps.movie_ids[seen['item_idx'].to_numpy()], 'title'].to_numpy(),
    'genres': movies.loc[split.maps.movie_ids[seen['item_idx'].to_numpy()], 'genres'].to_numpy(),
    'rating': seen['rating'].to_numpy(),
}))